# Test Benchmark — Mini protocole

Protocole réduit pour valider le pipeline avant le benchmark complet.

| Paramètre | Valeur |
|---|---|
| Images profiling | 30 (640×640) |
| Images MAP | 10 (résolution originale) |
| Warmup | 5 itérations |
| Mesure | 10 itérations |
| Seed | 42 |

**Deux modes distincts :**
- **Profiling** → `load_model()` + `preprocess()` : images 640×640, comparaison de vitesse standardisée
- **MAP eval** → `load_model_eval()` + `evaluate_map()` : résolution native du modèle, conforme COCO

## 1. Configuration

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import torch
import numpy as np
import pandas as pd
from pathlib import Path

IMG_DIR     = "datasets/coco/val2017"
ANN_FILE    = "datasets/coco/annotations/instances_val2017.json"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_PROFILE_IMAGES = 300
N_EVAL_IMAGES    = 10
N_WARMUP         = 5
N_MEASURE        = 10
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM   : 8.5 GB


## 2. Chargement des données

- `load_profiling_data` → images redimensionnées à 640×640
- `load_eval_data` → images à résolution originale COCO (variable)

In [2]:
from utils.data_loader import load_profiling_data, load_eval_data

profile_data        = load_profiling_data(IMG_DIR, ANN_FILE, n=N_PROFILE_IMAGES, seed=SEED)
eval_data, coco_gt  = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL_IMAGES, seed=SEED)

print(profile_data)           # LazySampleList(n=30, ...)
print(f"Premier sample profiling : {profile_data[0]}")
print(f"Premier sample eval      : {eval_data[0]}")

loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
loading annotations into memory...
Done (t=0.54s)
creating index...
index created!
LazySampleList(n=300, img_dir='datasets/coco/val2017')
Premier sample profiling : {'image_id': 107554, 'path': 'datasets\\coco\\val2017\\000000107554.jpg', 'orig_size': (480, 640)}
Premier sample eval      : {'image_id': 107554, 'path': 'datasets\\coco\\val2017\\000000107554.jpg', 'orig_size': (480, 640)}


## 3. Fonctions utilitaires

In [3]:
from utils.benchmark import benchmark_model, ModuleBenchmark

# Accumulateurs pour les CSV finaux
profiling_rows, map_rows = [], []

def record(name, result, maps):
    profiling_rows.append({"model": name, **{k: v for k, v in result.items() if k != "modules"}})
    map_rows.append({"model": name, **maps})
    print(f"  Benchmark (640×640) : {result['mean_ms']:.1f} ms ± {result['std_ms']:.1f}  ({result['fps']:.1f} FPS)")
    print(f"  MAP (native res)    : AP={maps.get('AP', -1):.3f}  AP50={maps.get('AP50', -1):.3f}")

print("OK — benchmark_model, ModuleBenchmark importés depuis utils/benchmark.py")

OK — benchmark_model, ModuleBenchmark importés depuis utils/benchmark.py


## 4. RetinaNet R50

In [4]:
import models.retinanet_r50 as r50

print("Chargement modèle benchmark...")
model_r50  = r50.load_model(DEVICE)
result_r50 = benchmark_model(model_r50, profile_data, r50.preprocess, r50.collate,
                             n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_r50; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_r50_eval = r50.load_model_eval(DEVICE)
map_r50        = r50.evaluate_map(model_r50_eval, eval_data, coco_gt, DEVICE)
del model_r50_eval; torch.cuda.empty_cache()

record("retinanet_r50", result_r50, map_r50)

Chargement modèle benchmark...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.14s).
Accumulating evaluation results...
DONE (t=0.09s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.525
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.820
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.535
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.305
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.546
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.728
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.428
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.537
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.573
 Avera

## 5. RetinaNet R101

In [5]:
import models.retinanet_r101 as r101

print("Chargement modèle benchmark...")
model_r101  = r101.load_model(DEVICE)
result_r101 = benchmark_model(model_r101, profile_data, r101.preprocess, r101.collate,
                              n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_r101; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_r101_eval = r101.load_model_eval(DEVICE)
map_r101        = r101.evaluate_map(model_r101_eval, eval_data, coco_gt, DEVICE)
del model_r101_eval; torch.cuda.empty_cache()

record("retinanet_r101", result_r101, map_r101)

Loading config d:\school\ProjectDSAI2026_2\detectron2\model_zoo\configs\COCO-Detection\../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Chargement modèle benchmark...


The checkpoint state_dict contains keys that are not used by the model:
  pixel_mean
  pixel_std
c:\ProgramData\anaconda3\Lib\site-packages\torch\functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4324.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading config d:\school\ProjectDSAI2026_2\detectron2\model_zoo\configs\COCO-Detection\../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Chargement eval model + MAP...


The checkpoint state_dict contains keys that are not used by the model:
  pixel_mean
  pixel_std


Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.474
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.708
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.203
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.708
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.371
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.519
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.573
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

## 6. FCOS R50

In [6]:
import models.fcos_r50 as fcos

print("Chargement modèle benchmark...")
model_fcos  = fcos.load_model(DEVICE)
result_fcos = benchmark_model(model_fcos, profile_data, fcos.preprocess, fcos.collate,
                              n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_fcos; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_fcos_eval = fcos.load_model_eval(DEVICE)
map_fcos        = fcos.evaluate_map(model_fcos_eval, eval_data, coco_gt, DEVICE)
del model_fcos_eval; torch.cuda.empty_cache()

record("fcos_r50", result_fcos, map_fcos)

Chargement modèle benchmark...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.08s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.514
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.764
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.366
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.533
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.674
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.541
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.569
 Avera

## 7. EfficientDet — test pipeline (PIL + F.normalize, sans +1)

In [7]:
# ── Test rapide D4 avec le pipeline PIL copié de l'ancien code ──────────────────
import importlib
import models.efficientdet_d4 as ed4
importlib.reload(ed4)

print("Chargement D4 eval model...")
model_ed4_test = ed4.load_model_eval(DEVICE)

print(f"Évaluation MAP sur {N_EVAL_IMAGES} images...")
map_ed4_test = ed4.evaluate_map(model_ed4_test, eval_data, coco_gt, DEVICE)
del model_ed4_test; torch.cuda.empty_cache()

print(f"\n>>> D4 AP  = {map_ed4_test['AP']:.4f}  (attendu ~0.43)")
print(f">>> D4 AP50= {map_ed4_test['AP50']:.4f}  (attendu ~0.62)")
print(f"\nRésultat complet : {map_ed4_test}")

Chargement D4 eval model...
Évaluation MAP sur 10 images...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.582
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.633
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.362
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.593
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.879
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.459
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.618
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.648
 Average

In [8]:
import models.efficientdet_d4 as ed4

print("Chargement modèle benchmark...")
model_ed4  = ed4.load_model(DEVICE)
result_ed4 = benchmark_model(model_ed4, profile_data, ed4.preprocess, ed4.collate,
                             n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed4; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed4_eval = ed4.load_model_eval(DEVICE)
map_ed4        = ed4.evaluate_map(model_ed4_eval, eval_data, coco_gt, DEVICE)
del model_ed4_eval; torch.cuda.empty_cache()

record("efficientdet_d4", result_ed4, map_ed4)

Chargement modèle benchmark...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.582
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.633
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.362
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.593
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.879
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.459
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.618
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.648
 Avera

## 8. EfficientDet D5

In [9]:
import models.efficientdet_d5 as ed5

print("Chargement modèle benchmark...")
model_ed5  = ed5.load_model(DEVICE)
result_ed5 = benchmark_model(model_ed5, profile_data, ed5.preprocess, ed5.collate,
                             n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed5; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed5_eval = ed5.load_model_eval(DEVICE)
map_ed5        = ed5.evaluate_map(model_ed5_eval, eval_data, coco_gt, DEVICE)
del model_ed5_eval; torch.cuda.empty_cache()

record("efficientdet_d5", result_ed5, map_ed5)

Chargement modèle benchmark...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.524
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.728
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.605
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.417
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.573
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.764
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.415
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.555
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.587
 Avera

## 9. EfficientDet D6

In [10]:
import models.efficientdet_d6 as ed6

print("Chargement modèle benchmark...")
model_ed6  = ed6.load_model(DEVICE)
result_ed6 = benchmark_model(model_ed6, profile_data, ed6.preprocess, ed6.collate,
                             n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed6; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed6_eval = ed6.load_model_eval(DEVICE)
map_ed6        = ed6.evaluate_map(model_ed6_eval, eval_data, coco_gt, DEVICE)
del model_ed6_eval; torch.cuda.empty_cache()

record("efficientdet_d6", result_ed6, map_ed6)

Chargement modèle benchmark...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.08s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.507
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.692
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.295
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.561
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.760
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.374
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.584
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.620
 Avera

## 10. Résultats

In [11]:
df_prof = pd.DataFrame(profiling_rows).set_index("model")
df_map  = pd.DataFrame(map_rows).set_index("model")

df_prof.to_csv(RESULTS_DIR / "profiling_test.csv")
df_map.to_csv(RESULTS_DIR  / "map_test.csv")

print("=== Profiling — 640×640 (ms) ===")
display(df_prof.round(2))

print("\n=== MAP — résolution native COCO ===")
display(df_map.round(3))

print(f"\nCSV sauvegardés dans {RESULTS_DIR.resolve()}")

=== Profiling — 640×640 (ms) ===


,mean_ms,std_ms,min_ms,max_ms,fps
model,,,,,
retinanet_r50,66.86,6.04,59.71,78.27,14.96
retinanet_r101,76.16,6.12,69.45,89.49,13.13
fcos_r50,64.07,14.82,55.72,107.38,15.61
efficientdet_d4,60.87,5.93,52.75,72.16,16.43
efficientdet_d5,71.45,10.91,60.70,93.84,13.99
efficientdet_d6,83.24,12.66,78.26,121.15,12.01



=== MAP — résolution native COCO ===


,AP,AP50,AP75,APs,APm,APl
model,,,,,,
retinanet_r50,0.525,0.820,0.535,0.305,0.546,0.728
retinanet_r101,0.474,0.708,0.494,0.203,0.520,0.708
fcos_r50,0.514,0.764,0.580,0.366,0.533,0.674
efficientdet_d4,0.582,0.738,0.633,0.362,0.593,0.879
efficientdet_d5,0.524,0.728,0.605,0.417,0.573,0.764
efficientdet_d6,0.507,0.692,0.532,0.295,0.561,0.760



CSV sauvegardés dans D:\school\ProjectDSAI2026_2\results


## 11. Profilage méso-scopique — PyTorch Profiler

Décomposition du forward pass par opération PyTorch (couche par couche).

Le profiler utilise son propre `schedule(wait=0, warmup=N_WARMUP, active=N_MEASURE)` :
- phase warmup : GPU chauffe, trace collectée mais non commitée
- phase active : trace exportée vers TensorBoard + Chrome JSON

**Sorties :**
```
results/profiler/pytorch/<model>/
  tensorboard/        ← tensorboard --logdir results/profiler/pytorch/<model>/tensorboard
  chrome_trace.json   ← chrome://tracing  ou  ui.perfetto.dev
  summary*.txt        ← tableaux triés par cuda_time_total
```

In [ ]:
from profiler.pytorch_profiler import profile_with_pytorch

PROF_OUTPUT  = "results/profiler/pytorch"
PROFILER_TAG = "baseline"

MODELS_TO_PROFILE = {
    "retinanet_r50":   (r50,  r50.load_model),
    "retinanet_r101":  (r101, r101.load_model),
    "fcos_r50":        (fcos, fcos.load_model),
    "efficientdet_d4": (ed4,  ed4.load_model),
    "efficientdet_d5": (ed5,  ed5.load_model),
    "efficientdet_d6": (ed6,  ed6.load_model),
}

pytorch_prof_results = {}

for model_name, (mod, load_fn) in MODELS_TO_PROFILE.items():
    print(f"\n{'─'*55}")
    print(f"  {model_name}  [tag={PROFILER_TAG}]")
    print(f"{'─'*55}")

    model = load_fn(DEVICE)
    result = profile_with_pytorch(
        model=model, data=profile_data,
        preprocess_fn=mod.preprocess, collate_fn=mod.collate,
        n_warmup=N_WARMUP, n_active=N_MEASURE,
        output_dir=PROF_OUTPUT, model_name=model_name,
        tag=PROFILER_TAG, device=DEVICE,
    )
    result["profiler"].export_chrome_trace(f"{PROF_OUTPUT}/{result['run_name']}/tensorboard/{result['run_name']}.pt.trace.json")
    pytorch_prof_results[model_name] = result
    del model; torch.cuda.empty_cache()

print(f"\nTraces dans : {PROF_OUTPUT}")
print(f"Ouvrir dans chrome://tracing ou ui.perfetto.dev")


───────────────────────────────────────────────────────
  retinanet_r50  [tag=baseline]
───────────────────────────────────────────────────────

  PyTorch Profiler — retinanet_r50__baseline__20260613_184058
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*        18.45%     210.041ms       100.00%        1.139s     113.866ms           0 B           0 B            10            --  
                                       aten::lift_fresh 


───────────────────────────────────────────────────────
  retinanet_r101  [tag=baseline]
───────────────────────────────────────────────────────


  pixel_mean
  pixel_std



  PyTorch Profiler — retinanet_r101__baseline__20260613_184113
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         8.83%      77.694ms       100.00%     879.563ms      87.956ms           0 B           0 B            10            --  
                                       aten::lift_fresh         0.01%      48.100us         0.01%      48.100us       1.603us           0 B           0 B            30            --  
                

## 12. Benchmark par module — `ModuleBenchmark`

Décomposition du temps GPU **module feuille par module feuille** (Conv2d, BatchNorm2d, ReLU, etc.)
via des CUDA Events attachés en hooks `forward_pre` / `forward`.

**Convention de nommage des sorties :**
```
results/benchmark/modules/<tag>/
    retinanet_r50.csv
    retinanet_r101.csv
    fcos_r50.csv
    ...
```

Changer `BENCH_TAG` pour distinguer les runs (baseline, compiled, tensorrt…).

In [13]:
BENCH_TAG = "baseline"   # changer ici : "compiled", "tensorrt", etc.

# ── Répertoire de sortie ───────────────────────────────────────────────────────
MODULES_OUT = Path("results/benchmark/modules") / BENCH_TAG
MODULES_OUT.mkdir(parents=True, exist_ok=True)

# ── Choix du modèle ───────────────────────────────────────────────────────────
TARGET_MODEL_NAME = "retinanet_r50"
TARGET_MOD        = r50
TARGET_LOAD_FN    = r50.load_model

print(f"Benchmark par module : {TARGET_MODEL_NAME}  [tag={BENCH_TAG}]")
print(f"  warmup={N_WARMUP}, mesure={N_MEASURE}\n")

model_mb = TARGET_LOAD_FN(DEVICE)
mb       = ModuleBenchmark()

result_mb = benchmark_model(model_mb, profile_data, TARGET_MOD.preprocess, TARGET_MOD.collate,
                            n_warmup=N_WARMUP, n_measure=N_MEASURE,
                            device=DEVICE, module_benchmark=mb)
del model_mb; torch.cuda.empty_cache()

# ── Résumé global ─────────────────────────────────────────────────────────────
df_modules  = result_mb["modules"]
sum_modules = df_modules["mean_ms"].sum()
print(f"  Global     : {result_mb['mean_ms']:.2f} ms ± {result_mb['std_ms']:.2f}  ({result_mb['fps']:.1f} FPS)")
print(f"  Σ modules  : {sum_modules:.2f} ms  (parallélisme = {sum_modules/result_mb['mean_ms']:.2f}x)")
print(f"  Nb modules : {len(df_modules)}\n")

# ── Sauvegarde CSV ────────────────────────────────────────────────────────────
csv_path = MODULES_OUT / f"{TARGET_MODEL_NAME}.csv"
df_modules.to_csv(csv_path, index=False)
print(f"  Sauvegardé → {csv_path}\n")

# ── Top 20 modules ────────────────────────────────────────────────────────────
print("Top 20 modules (mean_ms décroissant) :")
display(df_modules.head(20)[["module", "type", "root_component", "mean_ms", "std_ms", "pct_sum", "n_samples"]].round(4))

# ── Agrégation par composant racine ───────────────────────────────────────────
print("\nTemps par composant :")
by_comp = (df_modules.groupby("root_component")["mean_ms"]
           .sum().sort_values(ascending=False).reset_index())
by_comp["pct_total"] = (by_comp["mean_ms"] / sum_modules * 100).round(2)
display(by_comp.round(4))

Benchmark par module : retinanet_r50  [tag=baseline]
  warmup=5, mesure=10

  Global     : 71.44 ms ± 10.29  (14.0 FPS)
  Σ modules  : 26.32 ms  (parallélisme = 0.37x)
  Nb modules : 160

  Sauvegardé → results\benchmark\modules\baseline\retinanet_r50.csv

Top 20 modules (mean_ms décroissant) :


,module,type,root_component,mean_ms,std_ms,pct_sum,n_samples
0,backbone.fpn.layer_blocks.0.0,0,backbone,1.3981,0.0041,5.31,10
1,backbone.body.conv1,conv1,backbone,0.9193,0.1211,3.49,10
2,transform,transform,transform,0.8616,0.4361,3.27,10
3,backbone.body.layer2.0.downsample.0,0,backbone,0.6762,0.0333,2.57,10
4,backbone.body.layer4.0.downsample.0,0,backbone,0.6665,0.0225,2.53,10
5,backbone.body.layer3.0.downsample.0,0,backbone,0.6422,0.0448,2.44,10
6,backbone.body.layer4.0.conv2,conv2,backbone,0.6272,0.0476,2.38,10
7,backbone.body.layer4.2.conv2,conv2,backbone,0.5690,0.0434,2.16,10
8,backbone.body.layer4.1.conv2,conv2,backbone,0.5471,0.0028,2.08,10
9,anchor_generator,anchor_generator,anchor_generator,0.4838,0.1550,1.84,10



Temps par composant :


,root_component,mean_ms,pct_total
0,backbone,22.8399,86.79
1,head,2.1312,8.10
2,transform,0.8616,3.27
3,anchor_generator,0.4838,1.84


In [14]:
# ── Benchmark par module sur tous les modèles ─────────────────────────────────
# BENCH_TAG défini dans la cellule précédente — changer là-bas pour un nouveau run
MODELS_TO_BENCH = {
    "retinanet_r50":   (r50,  r50.load_model),
    "retinanet_r101":  (r101, r101.load_model),
    "fcos_r50":        (fcos, fcos.load_model),
    "efficientdet_d4": (ed4,  ed4.load_model),
    "efficientdet_d5": (ed5,  ed5.load_model),
    "efficientdet_d6": (ed6,  ed6.load_model),
}

all_modules = {}
all_globals = {}

for mname, (mod, load_fn) in MODELS_TO_BENCH.items():
    print(f"  {mname}…", end=" ", flush=True)
    m  = load_fn(DEVICE)
    mb = ModuleBenchmark()
    res = benchmark_model(m, profile_data, mod.preprocess, mod.collate,
                          n_warmup=N_WARMUP, n_measure=N_MEASURE,
                          device=DEVICE, module_benchmark=mb)
    all_modules[mname] = res["modules"]
    all_globals[mname] = {k: v for k, v in res.items() if k != "modules"}
    del m; torch.cuda.empty_cache()

    csv_path = MODULES_OUT / f"{mname}.csv"
    res["modules"].to_csv(csv_path, index=False)
    print(f"{res['mean_ms']:.1f} ms  ({res['fps']:.1f} FPS)  — {len(res['modules'])} modules  → {csv_path}")

print(f"\nCSV sauvegardés dans : {MODULES_OUT.resolve()}")

print("\n=== Top 5 modules les plus lents par modèle ===")
for mname, df in all_modules.items():
    print(f"\n{mname} (global={all_globals[mname]['mean_ms']:.1f} ms):")
    display(df.head(5)[["module", "mean_ms", "pct_sum"]].round(4))

  retinanet_r50… 96.0 ms  (10.4 FPS)  — 160 modules  → results\benchmark\modules\baseline\retinanet_r50.csv
  retinanet_r101… 

  pixel_mean
  pixel_std


82.6 ms  (12.1 FPS)  — 130 modules  → results\benchmark\modules\baseline\retinanet_r101.csv
  fcos_r50… 62.7 ms  (15.9 FPS)  — 161 modules  → results\benchmark\modules\baseline\fcos_r50.csv
  efficientdet_d4… 123.1 ms  (8.1 FPS)  — 898 modules  → results\benchmark\modules\baseline\efficientdet_d4.csv
  efficientdet_d5… 148.7 ms  (6.7 FPS)  — 1000 modules  → results\benchmark\modules\baseline\efficientdet_d5.csv
  efficientdet_d6… 156.0 ms  (6.4 FPS)  — 1155 modules  → results\benchmark\modules\baseline\efficientdet_d6.csv

CSV sauvegardés dans : D:\school\ProjectDSAI2026_2\results\benchmark\modules\baseline

=== Top 5 modules les plus lents par modèle ===

retinanet_r50 (global=96.0 ms):


,module,mean_ms,pct_sum
0,backbone.fpn.layer_blocks.0.0,1.3593,4.00
1,anchor_generator,1.2605,3.71
2,transform,1.0764,3.17
3,backbone.body.conv1,0.9714,2.86
4,backbone.body.layer4.0.downsample.0,0.7338,2.16



retinanet_r101 (global=82.6 ms):


...

,module,mean_ms,pct_sum
0,model.backbone.blocks.1.0.conv_dw,1.1077,1.83
1,model.backbone.blocks.0.0.conv_dw,0.5230,0.87
2,model.backbone.blocks.1.2.conv_dw,0.4996,0.83
3,model.backbone.blocks.1.3.conv_dw,0.4991,0.83
4,model.backbone.blocks.1.1.conv_dw,0.4988,0.83



efficientdet_d5 (global=148.7 ms):


,module,mean_ms,pct_sum
0,model.backbone.blocks.1.0.conv_dw,1.1193,1.48
1,model.backbone.blocks.1.4.conv_dw,0.6422,0.85
2,model.backbone.blocks.1.3.conv_dw,0.6393,0.85
3,model.backbone.blocks.1.2.conv_dw,0.6382,0.85
4,model.backbone.blocks.1.1.conv_dw,0.6371,0.84



efficientdet_d6 (global=156.0 ms):


,module,mean_ms,pct_sum
0,model.backbone.blocks.1.0.conv_dw,1.4200,1.62
1,model.backbone.blocks.1.5.conv_dw,0.6603,0.75
2,model.backbone.blocks.1.1.conv_dw,0.6363,0.72
3,model.backbone.blocks.1.3.conv_dw,0.6349,0.72
4,model.backbone.blocks.1.4.conv_dw,0.6348,0.72
